# SP, DP, EMT Validation — 4th Order VBR SynGen with All Controllers (SMIB Fault)

Pure Python API (`dpsimpy`) notebook. Validates the 4th order VBR synchronous generator with:
- Exciter: `ExciterDC1Simp` — validated against PSAT reference
- Exciter + PSS: `PSS1A` — smoke test (no PSAT reference for PSS)
- Governor: `TurbineGovernorType1` — validated against PSAT reference
- Steam governor: `SteamTurbineGovernor + SteamTurbine` — smoke test
- Hydro governor: `HydroTurbineGovernor + HydroTurbine` — smoke test

Cross-validation:
- **Section 6**: SP / DP / EMT overlay for the Exciter case — visual comparison + RMSE
- **Section 7**: All controllers active simultaneously in SP, DP, and EMT (9 smoke tests)

**Scenario:** SMIB fault — 100 ms three-phase fault at t=1.0 s, cleared at t=1.1 s.

## Setup

In [ ]:
import sys
import os
import math
import numpy as np
import urllib.request
import matplotlib.pyplot as plt

import dpsimpy
from villas.dataprocessing.readtools import read_timeseries_dpsim
from villas.dataprocessing.timeseries import TimeSeries

In [ ]:
# Download PSAT reference files
PSAT_URL = "https://raw.githubusercontent.com/dpsim-simulator/reference-results/master/PSAT/SMIB-Fault/"
REF_DIR = "reference-results"
os.makedirs(REF_DIR, exist_ok=True)


def fetch_psat(filename):
    local = os.path.join(REF_DIR, filename)
    urllib.request.urlretrieve(PSAT_URL + filename, local)
    return read_timeseries_dpsim(local)


ts_psat_exciter = fetch_psat("PSAT_4OrderSyGen_withExciter_SMIB_Fault_100mS_TS_1mS.txt")
ts_psat_tgov1 = fetch_psat("PSAT_4OrderSyGen_withGovernor_SMIB_Fault_100mS_TS_1mS.txt")

In [ ]:
def calculate_variables_psat(ts, Xd_t, Xq_t):
    angle = ts["delta"].values - ts["theta_bus1"].values
    V = ts["V_Bus1"].values
    t = ts["delta"].time
    vd = V * np.sin(angle)
    vq = V * np.cos(angle)
    id_ = (ts["eq_t"].values - vq) / Xd_t
    iq_ = -(ts["ed_t"].values - vd) / Xq_t
    ts["vd"] = TimeSeries("vd", t, vd)
    ts["vq"] = TimeSeries("vq", t, vq)
    ts["id"] = TimeSeries("id", t, id_)
    ts["iq"] = TimeSeries("iq", t, iq_)
    return ts


ts_psat_exciter = calculate_variables_psat(ts_psat_exciter, Xd_t=0.2999, Xq_t=0.6500)
ts_psat_tgov1 = calculate_variables_psat(ts_psat_tgov1, Xd_t=0.2999, Xq_t=0.6500)

In [ ]:
# Nominal values and plot metadata for RMSE normalisation
nominal_values = {
    "Te": 0.540541,
    "Edq0_0": 0.403062,  # Ed'
    "Edq0_1": 0.944802,  # Eq'
    "Tm": 0.540541,
    "Vdq0_0": 0.639089,  # Vd
    "Vdq0_1": 0.833106,  # Vq
    "Idq0_0": 0.372443,  # Id
    "Idq0_1": 0.363119,  # Iq
    "delta": 0.909230,
    "w_r": 1.0,
    "Ef": 1.398800,
}

titles = {
    "Te": "Electrical Torque",
    "Edq0_0": "Ed'",
    "Edq0_1": "Eq'",
    "Tm": "Mechanical Torque",
    "Vdq0_0": "Vd",
    "Vdq0_1": "Vq",
    "Idq0_0": "Id",
    "Idq0_1": "Iq",
    "delta": "Rotor angle",
    "w_r": "Mechanical omega",
    "Ef": "Field voltage",
}

ylabels = {
    "Te": "Te (p.u.)",
    "Edq0_0": "Ed' (p.u.)",
    "Edq0_1": "Eq' (p.u.)",
    "Tm": "Tm (p.u.)",
    "Vdq0_0": "Vd (p.u.)",
    "Vdq0_1": "Vq (p.u.)",
    "Idq0_0": "Id (p.u.)",
    "Idq0_1": "Iq (p.u.)",
    "delta": "delta (rad)",
    "w_r": "w_r (p.u.)",
    "Ef": "Ef (p.u.)",
}

In [ ]:
T_STEP_COMMON = 1e-3
T_BEGIN = 0.0
T_END = 10.0


def plot_variable(varname_dpsim, varname_psat, ts_dpsim, ts_psat):
    """Plot DPsim vs PSAT, return RMSE in percent of nominal."""
    n = int((T_END - T_BEGIN) / T_STEP_COMMON)
    time = np.linspace(T_BEGIN, T_END, n)
    dv = ts_dpsim[varname_dpsim].interpolate(T_STEP_COMMON).values[:n]
    pv = ts_psat[varname_psat].interpolate(T_STEP_COMMON).values[:n]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(time, dv, label="DPsim")
    ax.plot(time, pv, "--", label="PSAT")
    ax.set_xlabel("time (s)")
    ax.set_ylabel(ylabels.get(varname_dpsim, varname_dpsim))
    ax.set_title(titles.get(varname_dpsim, varname_dpsim))
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()
    nom = nominal_values.get(varname_dpsim, 1.0)
    rmse = np.sqrt(np.mean((dv - pv) ** 2)) / nom * 100
    print(f"RMSE {varname_dpsim}: {rmse:.3f}% of nominal {nom:.4f} pu")
    return rmse


def plot_node_volt_abs(varname_dpsim, varname_psat, ts_dpsim, ts_psat, nom_voltage):
    """Plot |v| (DPsim, pu) vs V (PSAT, pu), return RMSE in percent."""
    n = int((T_END - T_BEGIN) / T_STEP_COMMON)
    time = np.linspace(T_BEGIN, T_END, n)
    dv = (
        ts_dpsim[varname_dpsim].interpolate(T_STEP_COMMON).abs().values[:n]
        / nom_voltage
    )
    pv = ts_psat[varname_psat].interpolate(T_STEP_COMMON).values[:n]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(time, dv, label="DPsim")
    ax.plot(time, pv, "--", label="PSAT")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("V (p.u.)")
    ax.set_title(f"|{varname_dpsim}|")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()
    rmse = np.sqrt(np.mean((dv - pv) ** 2)) * 100
    print(f"RMSE |{varname_dpsim}|: {rmse:.3f}%")
    return rmse


def plot_node_volt_angle(varname_dpsim, varname_psat, ts_dpsim, ts_psat, nom_voltage):
    """Plot voltage angle (DPsim, deg) vs PSAT angle (rad→deg), RMSE post-fault."""
    n = int((T_END - T_BEGIN) / T_STEP_COMMON)
    time = np.linspace(T_BEGIN, T_END, n)
    dv = ts_dpsim[varname_dpsim].interpolate(T_STEP_COMMON).phase().values[:n]
    pv = ts_psat[varname_psat].interpolate(T_STEP_COMMON).values[:n] / math.pi * 180
    post = int(1.1 / T_STEP_COMMON)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(time, dv, label="DPsim")
    ax.plot(time, pv, "--", label="PSAT")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("angle (deg)")
    ax.set_title(f"angle({varname_dpsim})")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()
    rmse = np.sqrt(np.mean((dv[post:] - pv[post:]) ** 2))
    print(f"RMSE post-fault angle {varname_dpsim}: {rmse:.6f} deg")
    return rmse

In [ ]:
# Parametrization — Kundur 4th order (P. Kundur, Power System Stability and Control, Ex. 3.2)
from math import pi

nom_power = 555e6
nom_voltage_mv = 24e3  # phase-to-phase RMS
frequency = 60.0
H = 3.7
Ld, Lq, L0 = 1.8099, 1.7600, 0.15
Ld_t, Lq_t = 0.2999, 0.6500
Td0_t, Tq0_t = 8.0669, 0.9991

# CIGRE benchmark line (100 km, 24 kV)
omega = 2 * pi * frequency
ratio = nom_voltage_mv / 230e3
line_resistance = 1.27e-4 * 529 * 100 * ratio**2
line_inductance = 9.05e-4 * 529 / omega * 100 * ratio**2
line_capacitance = (1.81e-3 / 529) / omega * 100 / ratio**2
line_conductance = 0.0

# Fault switch impedance (short = 0.001 * Zbase)
Zbase = nom_voltage_mv**2 / nom_power
switch_open = 1e6
switch_closed = 0.001 * Zbase

# ExciterDC1Simp  — Kundur / Milano (2010) p.363
EXC_Ka, EXC_Ta = 46.0, 0.06
EXC_Kef, EXC_Tef = -0.0435, 0.46
EXC_Kf, EXC_Tf = 0.1, 1.0
EXC_Tr = 0.02
EXC_Aef, EXC_Bef = 0.33, 0.1
EXC_MaxVa, EXC_MinVa = 1.0, -0.9

# PSS1A — Kundur typical
PSS_Kp, PSS_Kv, PSS_Kw = 0.0, 0.0, 20.0
PSS_T1, PSS_T2 = 0.14, 0.04
PSS_T3, PSS_T4 = 0.14, 0.04
PSS_Vs_max, PSS_Vs_min = 0.1, -0.1
PSS_Tw = 10.0

# TurbineGovernorType1 — Kundur / Milano (2010)
TG_OmRef = 1.0
TG_R = 0.02
TG_Tmax, TG_Tmin = 1.2, 0.3
TG_Ts, TG_Tc = 0.1, 0.45
TG_T3, TG_T4, TG_T5 = 0.0, 12.0, 50.0

# SteamTurbineGovernor + SteamTurbine — from Examples.h
SG_R, SG_T1, SG_T2, SG_T3 = 0.04, 0.0, 0.2, 0.1
SG_dPmax, SG_dPmin = 50.0, -50.0
SG_Pmax, SG_Pmin, SG_OmRef, SG_Kbc = 1.0, 0.0, 1.0, 0.0
ST_Fhp, ST_Fip, ST_Flp = 0.3, 0.3, 0.4
ST_Tch, ST_Trh, ST_Tco = 0.1, 4.0, 0.3

# HydroTurbineGovernor + HydroTurbine — from Examples.h
HG_R, HG_T1, HG_T2, HG_T3 = 0.05, 1.0, 0.2, 0.5
HG_Pmax, HG_Pmin, HG_OmRef = 1.0, 0.0, 1.0
HT_Tw = 1.0

In [ ]:
# Power flow (SP, NRP solver — used to initialise all domain simulations)
set_point_voltage = 1.05 * nom_voltage_mv

sim_name_pf = "SP_ReducedOrderSG_SMIB_PF"
dpsimpy.Logger.set_log_dir("logs/" + sim_name_pf)

n1_pf = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single)
n2_pf = dpsimpy.sp.SimNode("n2", dpsimpy.PhaseType.Single)

gen_pf = dpsimpy.sp.ph1.SynchronGenerator("gen", dpsimpy.LogLevel.off)
gen_pf.set_parameters(
    rated_apparent_power=nom_power,
    rated_voltage=nom_voltage_mv,
    set_point_active_power=300e6,
    set_point_voltage=set_point_voltage,
    powerflow_bus_type=dpsimpy.PowerflowBusType.PV,
)
gen_pf.set_base_voltage(nom_voltage_mv)
gen_pf.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PV)

extnet_pf = dpsimpy.sp.ph1.NetworkInjection("slack", dpsimpy.LogLevel.off)
extnet_pf.set_parameters(nom_voltage_mv)
extnet_pf.set_base_voltage(nom_voltage_mv)
extnet_pf.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.VD)

line_pf = dpsimpy.sp.ph1.PiLine("PiLine", dpsimpy.LogLevel.off)
line_pf.set_parameters(
    R=line_resistance, L=line_inductance, C=line_capacitance, G=line_conductance
)
line_pf.set_base_voltage(nom_voltage_mv)

gen_pf.connect([n1_pf])
line_pf.connect([n1_pf, n2_pf])
extnet_pf.connect([n2_pf])

system_pf = dpsimpy.SystemTopology(
    frequency, [n1_pf, n2_pf], [gen_pf, line_pf, extnet_pf]
)

logger_pf = dpsimpy.Logger(sim_name_pf)
logger_pf.log_attribute("n1.v", "v", n1_pf)
logger_pf.log_attribute("n2.v", "v", n2_pf)

sim_pf = dpsimpy.Simulation(sim_name_pf, dpsimpy.LogLevel.off)
sim_pf.set_system(system_pf)
sim_pf.set_domain(dpsimpy.Domain.SP)
sim_pf.set_solver(dpsimpy.Solver.NRP)
sim_pf.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
sim_pf.set_time_step(0.1)
sim_pf.set_final_time(0.5)
sim_pf.add_logger(logger_pf)
sim_pf.run()

In [ ]:
def smib_fault(
    sim_pf,
    gen_pf,
    domain="SP",
    name_suffix="",
    with_exciter=False,
    with_pss=False,
    with_governor_type1=False,
    with_steam_governor=False,
    with_hydro_governor=False,
    final_time=10.0,
    time_step=1e-3,
):
    """Build, run, and log an SMIB fault simulation for any domain.

    Returns the simulation log name (used to locate the CSV).
    """
    if with_pss and not with_exciter:
        raise ValueError("PSS requires an exciter — set with_exciter=True")

    name = f"{domain}_SynGen4Order_VBR_SMIB_Fault{name_suffix}"
    dpsimpy.Logger.set_log_dir("logs/" + name)

    # ------------------------------------------------------------------
    # Domain-specific classes
    # ------------------------------------------------------------------
    if domain == "SP":
        SimNode = dpsimpy.sp.SimNode
        gnd = dpsimpy.sp.SimNode.gnd
        SynGen4 = dpsimpy.sp.ph1.SynchronGenerator4OrderVBR
        PiLine_cls = dpsimpy.sp.ph1.PiLine
        Switch_cls = dpsimpy.sp.ph1.Switch
        Slack_cls = dpsimpy.sp.ph1.NetworkInjection
        ph_type = dpsimpy.PhaseType.Single
        dom = dpsimpy.Domain.SP
        edq_attr = "Edq_t"

        def sw_event(t, sw, state):
            return dpsimpy.event.SwitchEvent(t, sw, state)

        three_phase = False
    elif domain == "DP":
        SimNode = dpsimpy.dp.SimNode
        gnd = dpsimpy.dp.SimNode.gnd
        SynGen4 = dpsimpy.dp.ph1.SynchronGenerator4OrderVBR
        PiLine_cls = dpsimpy.dp.ph1.PiLine
        Switch_cls = dpsimpy.dp.ph1.Switch
        Slack_cls = dpsimpy.dp.ph1.NetworkInjection
        ph_type = dpsimpy.PhaseType.Single
        dom = dpsimpy.Domain.DP
        edq_attr = "Edq0_t"

        def sw_event(t, sw, state):
            return dpsimpy.event.SwitchEvent(t, sw, state)

        three_phase = False
    elif domain == "EMT":
        SimNode = dpsimpy.emt.SimNode
        gnd = dpsimpy.emt.SimNode.gnd
        SynGen4 = dpsimpy.emt.ph3.SynchronGenerator4OrderVBR
        PiLine_cls = dpsimpy.emt.ph3.PiLine
        Switch_cls = dpsimpy.emt.ph3.Switch
        Slack_cls = dpsimpy.emt.ph3.NetworkInjection
        ph_type = dpsimpy.PhaseType.ABC
        dom = dpsimpy.Domain.EMT
        edq_attr = "Edq0_t"

        def sw_event(t, sw, state):
            return dpsimpy.event.SwitchEvent3Ph(t, sw, state)

        three_phase = True
    else:
        raise ValueError(f"Unknown domain '{domain}'. Use 'SP', 'DP', or 'EMT'.")

    # ------------------------------------------------------------------
    # Nodes
    # ------------------------------------------------------------------
    n1 = SimNode("n1", ph_type)
    n2 = SimNode("n2", ph_type)

    # ------------------------------------------------------------------
    # Generator
    # ------------------------------------------------------------------
    gen = SynGen4("gen", dpsimpy.LogLevel.off)
    gen.set_operational_parameters_per_unit(
        nom_power, nom_voltage_mv, frequency, H, Ld, Lq, L0, Ld_t, Lq_t, Td0_t, Tq0_t
    )

    # ------------------------------------------------------------------
    # Controllers
    # ------------------------------------------------------------------
    if with_exciter:
        exciter = dpsimpy.signal.ExciterDC1Simp("Exciter")
        ep = dpsimpy.signal.ExciterDC1SimpParameters()
        ep.Ka = EXC_Ka
        ep.Ta = EXC_Ta
        ep.Kef = EXC_Kef
        ep.Tef = EXC_Tef
        ep.Kf = EXC_Kf
        ep.Tf = EXC_Tf
        ep.Tr = EXC_Tr
        ep.Aef = EXC_Aef
        ep.Bef = EXC_Bef
        ep.MaxVa = EXC_MaxVa
        ep.MinVa = EXC_MinVa
        exciter.set_parameters(ep)
        gen.add_exciter(exciter)

    if with_pss:
        pss = dpsimpy.signal.PSS1A("PSS")
        pp = dpsimpy.signal.PSS1AParameters()
        pp.Kp = PSS_Kp
        pp.Kv = PSS_Kv
        pp.Kw = PSS_Kw
        pp.T1 = PSS_T1
        pp.T2 = PSS_T2
        pp.T3 = PSS_T3
        pp.T4 = PSS_T4
        pp.Vs_max = PSS_Vs_max
        pp.Vs_min = PSS_Vs_min
        pp.Tw = PSS_Tw
        pss.set_parameters(pp)
        gen.add_pss(pss)

    if with_governor_type1:
        gov = dpsimpy.signal.TurbineGovernorType1("TurbGov")
        gov.set_parameters(
            TG_T3, TG_T4, TG_T5, TG_Tc, TG_Ts, TG_R, TG_Tmin, TG_Tmax, TG_OmRef
        )
        gen.add_governor(gov)

    if with_steam_governor:
        sg = dpsimpy.signal.SteamTurbineGovernor("SteamGov")
        sgp = dpsimpy.signal.SteamGovernorParameters()
        sgp.R = SG_R
        sgp.T1 = SG_T1
        sgp.T2 = SG_T2
        sgp.T3 = SG_T3
        sgp.dPmax = SG_dPmax
        sgp.dPmin = SG_dPmin
        sgp.Pmax = SG_Pmax
        sgp.Pmin = SG_Pmin
        sgp.OmRef = SG_OmRef
        sgp.Kbc = SG_Kbc
        sg.set_parameters(sgp)
        st = dpsimpy.signal.SteamTurbine("SteamTurbine")
        stp = dpsimpy.signal.SteamTurbineParameters()
        stp.Fhp = ST_Fhp
        stp.Fip = ST_Fip
        stp.Flp = ST_Flp
        stp.Tch = ST_Tch
        stp.Trh = ST_Trh
        stp.Tco = ST_Tco
        st.set_parameters(stp)
        gen.add_governor_and_turbine(sg, st)

    if with_hydro_governor:
        hg = dpsimpy.signal.HydroTurbineGovernor("HydroGov")
        hgp = dpsimpy.signal.HydroGovernorParameters()
        hgp.R = HG_R
        hgp.T1 = HG_T1
        hgp.T2 = HG_T2
        hgp.T3 = HG_T3
        hgp.Pmax = HG_Pmax
        hgp.Pmin = HG_Pmin
        hgp.OmRef = HG_OmRef
        hg.set_parameters(hgp)
        ht = dpsimpy.signal.HydroTurbine("HydroTurbine")
        htp = dpsimpy.signal.HydroTurbineParameters()
        htp.Tw = HT_Tw
        ht.set_parameters(htp)
        gen.add_governor_and_turbine(hg, ht)

    # ------------------------------------------------------------------
    # Network components
    # ------------------------------------------------------------------
    if three_phase:
        to3 = dpsimpy.Math.single_phase_parameter_to_three_phase
        pi_line = PiLine_cls("PiLine", dpsimpy.LogLevel.off)
        pi_line.set_parameters(
            to3(line_resistance),
            to3(line_inductance),
            to3(line_capacitance),
            to3(line_conductance),
        )
        switch = Switch_cls("Fault", dpsimpy.LogLevel.off)
        switch.set_parameters(to3(switch_open), to3(switch_closed))
        slack = Slack_cls("slack", dpsimpy.LogLevel.off)
        # EMT NetworkInjection is fully initialised by init_with_powerflow
    else:
        pi_line = PiLine_cls("PiLine", dpsimpy.LogLevel.off)
        pi_line.set_parameters(
            line_resistance, line_inductance, line_capacitance, line_conductance
        )
        switch = Switch_cls("Fault", dpsimpy.LogLevel.off)
        switch.set_parameters(switch_open, switch_closed)
        slack = Slack_cls("slack", dpsimpy.LogLevel.off)
        slack.set_parameters(V_ref=nom_voltage_mv)
    switch.open()

    # ------------------------------------------------------------------
    # Topology and power-flow initialisation
    # ------------------------------------------------------------------
    gen.connect([n1])
    switch.connect([gnd, n1])
    pi_line.connect([n1, n2])
    slack.connect([n2])

    system = dpsimpy.SystemTopology(frequency, [n1, n2], [gen, pi_line, slack, switch])
    system.init_with_powerflow(system_pf, dom)

    # ------------------------------------------------------------------
    # Logging
    # ------------------------------------------------------------------
    logger = dpsimpy.Logger(name)
    logger.log_attribute("n1.v", "v", n1)
    logger.log_attribute("n2.v", "v", n2)
    logger.log_attribute("Te", "Te", gen)
    logger.log_attribute("Tm", "Tm", gen)
    logger.log_attribute("delta", "delta", gen)
    logger.log_attribute("w_r", "w_r", gen)
    logger.log_attribute("Vdq0", "Vdq0", gen)
    logger.log_attribute("Idq0", "Idq0", gen)
    logger.log_attribute("Edq0", edq_attr, gen)
    if with_exciter:
        logger.log_attribute("Ef", "Ef", gen)

    # ------------------------------------------------------------------
    # Simulation
    # ------------------------------------------------------------------
    sim = dpsimpy.Simulation(name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.do_init_from_nodes_and_terminals(True)
    sim.set_domain(dom)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.do_system_matrix_recomputation(True)
    sim.add_event(sw_event(1.0, switch, True))
    sim.add_event(sw_event(1.1, switch, False))
    sim.add_logger(logger)
    sim.run()

    return name

## Section 1 — SP: 4th Order + ExciterDC1Simp vs PSAT

In [ ]:
log_exc = smib_fault(
    sim_pf,
    gen_pf,
    domain="SP",
    name_suffix="_withExciter",
    with_exciter=True,
    final_time=10.0,
)

csv_path = f"logs/{log_exc}/{log_exc}.csv"
ts_sp_exc = read_timeseries_dpsim(csv_path)

In [ ]:
rmse_Te = plot_variable("Te", "p", ts_sp_exc, ts_psat_exciter)
rmse_Ef = plot_variable("Ef", "vf", ts_sp_exc, ts_psat_exciter)
rmse_Ed = plot_variable("Edq0_0", "ed_t", ts_sp_exc, ts_psat_exciter)
rmse_Eq = plot_variable("Edq0_1", "eq_t", ts_sp_exc, ts_psat_exciter)
rmse_Vd = plot_variable("Vdq0_0", "vd", ts_sp_exc, ts_psat_exciter)
rmse_Vq = plot_variable("Vdq0_1", "vq", ts_sp_exc, ts_psat_exciter)
rmse_Id = plot_variable("Idq0_0", "id", ts_sp_exc, ts_psat_exciter)
rmse_Iq = plot_variable("Idq0_1", "iq", ts_sp_exc, ts_psat_exciter)
rmse_delta = plot_variable("delta", "delta", ts_sp_exc, ts_psat_exciter)
rmse_wr = plot_variable("w_r", "omega", ts_sp_exc, ts_psat_exciter)
rmse_v1 = plot_node_volt_abs(
    "n1.v", "V_Bus1", ts_sp_exc, ts_psat_exciter, nom_voltage_mv
)
rmse_v1a = plot_node_volt_angle(
    "n1.v", "theta_bus1", ts_sp_exc, ts_psat_exciter, nom_voltage_mv
)

In [ ]:
tol = 2.0  # %
tol_Id = 12.0  # % (exciter initialization transient causes larger Id mismatch vs PSAT)
tol_Iq = 5.0  # %
tol_v1 = 0.2  # deg

assert rmse_Te < tol, f"Te RMSE {rmse_Te:.3f}% > {tol}%"
assert rmse_Ef < tol, f"Ef RMSE {rmse_Ef:.3f}% > {tol}%"
assert rmse_Ed < tol, f"Ed' RMSE {rmse_Ed:.3f}% > {tol}%"
assert rmse_Eq < tol, f"Eq' RMSE {rmse_Eq:.3f}% > {tol}%"
assert rmse_Vd < tol, f"Vd RMSE {rmse_Vd:.3f}% > {tol}%"
assert rmse_Vq < tol, f"Vq RMSE {rmse_Vq:.3f}% > {tol}%"
assert rmse_Id < tol_Id, f"Id RMSE {rmse_Id:.3f}% > {tol_Id}%"
assert rmse_Iq < tol_Iq, f"Iq RMSE {rmse_Iq:.3f}% > {tol_Iq}%"
assert rmse_delta < tol, f"delta RMSE {rmse_delta:.3f}% > {tol}%"
assert rmse_wr < tol, f"w_r RMSE {rmse_wr:.3f}% > {tol}%"
assert rmse_v1a < tol_v1, f"n1.v angle RMSE {rmse_v1a:.6f} deg > {tol_v1} deg"
print("Section 1 assertions passed.")

## Section 2 — SP: 4th Order + ExciterDC1Simp + PSS1A (smoke test)

No PSAT reference available for PSS. Asserts that the simulation runs stably: `Ef` finite,
`w_r` returns to within 0.5% of 1.0 pu by t=10 s.

In [ ]:
log_pss = smib_fault(
    sim_pf,
    gen_pf,
    domain="SP",
    name_suffix="_withPSS",
    with_exciter=True,
    with_pss=True,
    final_time=10.0,
)

ts_sp_pss = read_timeseries_dpsim(f"logs/{log_pss}/{log_pss}.csv")

Ef_final = ts_sp_pss["Ef"].values[-1]
wr_final = ts_sp_pss["w_r"].values[-1]

assert math.isfinite(Ef_final) and Ef_final > 0, f"Ef not positive finite: {Ef_final}"
assert (
    abs(wr_final - 1.0) < 0.005
), f"w_r {wr_final:.4f} pu not within 0.5% of 1.0 at t=10s"
print(f"PSS smoke test passed — Ef={Ef_final:.4f} pu, w_r={wr_final:.6f} pu at t=10s")

## Section 3 — SP: 4th Order + TurbineGovernorType1 vs PSAT

In [ ]:
log_tg = smib_fault(
    sim_pf,
    gen_pf,
    domain="SP",
    name_suffix="_withTGov1",
    with_governor_type1=True,
    final_time=10.0,
)

ts_sp_tg = read_timeseries_dpsim(f"logs/{log_tg}/{log_tg}.csv")

In [ ]:
rmse_Te_tg = plot_variable("Te", "p", ts_sp_tg, ts_psat_tgov1)
rmse_Tm_tg = plot_variable("Tm", "pm", ts_sp_tg, ts_psat_tgov1)
rmse_Ed_tg = plot_variable("Edq0_0", "ed_t", ts_sp_tg, ts_psat_tgov1)
rmse_Eq_tg = plot_variable("Edq0_1", "eq_t", ts_sp_tg, ts_psat_tgov1)
rmse_Vd_tg = plot_variable("Vdq0_0", "vd", ts_sp_tg, ts_psat_tgov1)
rmse_Vq_tg = plot_variable("Vdq0_1", "vq", ts_sp_tg, ts_psat_tgov1)
rmse_Id_tg = plot_variable("Idq0_0", "id", ts_sp_tg, ts_psat_tgov1)
rmse_Iq_tg = plot_variable("Idq0_1", "iq", ts_sp_tg, ts_psat_tgov1)
rmse_delta_tg = plot_variable("delta", "delta", ts_sp_tg, ts_psat_tgov1)
rmse_wr_tg = plot_variable("w_r", "omega", ts_sp_tg, ts_psat_tgov1)
rmse_v1_tg = plot_node_volt_abs(
    "n1.v", "V_Bus1", ts_sp_tg, ts_psat_tgov1, nom_voltage_mv
)
rmse_v1a_tg = plot_node_volt_angle(
    "n1.v", "theta_bus1", ts_sp_tg, ts_psat_tgov1, nom_voltage_mv
)

In [ ]:
tol_tg = 2.0  # %
tol_Id_tg = 15.0  # %
tol_Iq_tg = 6.0  # %
tol_v1_tg = 0.2  # deg

assert rmse_Te_tg < tol_tg, f"Te RMSE {rmse_Te_tg:.3f}% > {tol_tg}%"
assert rmse_Tm_tg < tol_tg, f"Tm RMSE {rmse_Tm_tg:.3f}% > {tol_tg}%"
assert rmse_Ed_tg < tol_tg, f"Ed' RMSE {rmse_Ed_tg:.3f}% > {tol_tg}%"
assert rmse_Eq_tg < tol_tg, f"Eq' RMSE {rmse_Eq_tg:.3f}% > {tol_tg}%"
assert rmse_Vd_tg < tol_tg, f"Vd RMSE {rmse_Vd_tg:.3f}% > {tol_tg}%"
assert rmse_Vq_tg < tol_tg, f"Vq RMSE {rmse_Vq_tg:.3f}% > {tol_tg}%"
assert rmse_Id_tg < tol_Id_tg, f"Id RMSE {rmse_Id_tg:.3f}% > {tol_Id_tg}%"
assert rmse_Iq_tg < tol_Iq_tg, f"Iq RMSE {rmse_Iq_tg:.3f}% > {tol_Iq_tg}%"
assert rmse_delta_tg < tol_tg, f"delta RMSE {rmse_delta_tg:.3f}% > {tol_tg}%"
assert rmse_wr_tg < tol_tg, f"w_r RMSE {rmse_wr_tg:.3f}% > {tol_tg}%"
assert (
    rmse_v1a_tg < tol_v1_tg
), f"n1.v angle RMSE {rmse_v1a_tg:.6f} deg > {tol_v1_tg} deg"
print("Section 3 assertions passed.")

## Section 4 — SP: 4th Order + SteamTurbineGovernor (smoke test)

No PSAT reference available. Asserts: `Tm` finite and positive, `w_r` within 0.5% of 1.0 pu at t=10 s.

In [ ]:
log_steam = smib_fault(
    sim_pf,
    gen_pf,
    domain="SP",
    name_suffix="_withSteamGov",
    with_steam_governor=True,
    final_time=10.0,
)

ts_sp_steam = read_timeseries_dpsim(f"logs/{log_steam}/{log_steam}.csv")

Tm_final = ts_sp_steam["Tm"].values[-1]
wr_final_s = ts_sp_steam["w_r"].values[-1]

assert math.isfinite(Tm_final) and Tm_final > 0, f"Tm not positive finite: {Tm_final}"
assert (
    abs(wr_final_s - 1.0) < 0.005
), f"w_r {wr_final_s:.4f} pu not within 0.5% of 1.0 at t=10s"
print(
    f"Steam smoke test passed — Tm={Tm_final:.4f} pu, w_r={wr_final_s:.6f} pu at t=10s"
)

## Section 5 — SP: 4th Order + HydroTurbineGovernor (smoke test)

No PSAT reference available. Same smoke-test asserts as Section 4.

In [ ]:
log_hydro = smib_fault(
    sim_pf,
    gen_pf,
    domain="SP",
    name_suffix="_withHydroGov",
    with_hydro_governor=True,
    final_time=10.0,
)

ts_sp_hydro = read_timeseries_dpsim(f"logs/{log_hydro}/{log_hydro}.csv")

Tm_final_h = ts_sp_hydro["Tm"].values[-1]
wr_final_h = ts_sp_hydro["w_r"].values[-1]

assert (
    math.isfinite(Tm_final_h) and Tm_final_h > 0
), f"Tm not positive finite: {Tm_final_h}"
assert (
    abs(wr_final_h - 1.0) < 0.005
), f"w_r {wr_final_h:.4f} pu not within 0.5% of 1.0 at t=10s"
print(
    f"Hydro smoke test passed — Tm={Tm_final_h:.4f} pu, w_r={wr_final_h:.6f} pu at t=10s"
)

## Section 6 — SP / DP / EMT Cross-Validation with Exciter

Runs the same SMIB-fault + ExciterDC1Simp scenario in all three domains and overlays the
results visually. Asserts that the mechanical states (`w_r`, `delta`) and exciter output
(`Ef`) agree across domains to within tight tolerances. Electrical torque (`Te`) is plotted
for reference but not asserted — the SP and DP/EMT formulations compute Te differently
during fast transients (static vs dynamic phasor), so domain-to-domain Te divergence is
expected and not a model defect.

In [ ]:
# SP result is already available from Section 1 (ts_sp_exc / log_exc)
# Run DP
log_dp = smib_fault(
    sim_pf,
    gen_pf,
    domain="DP",
    name_suffix="_withExciter",
    with_exciter=True,
    final_time=10.0,
)
ts_dp = read_timeseries_dpsim(f"logs/{log_dp}/{log_dp}.csv")

# Run EMT
log_emt = smib_fault(
    sim_pf,
    gen_pf,
    domain="EMT",
    name_suffix="_withExciter",
    with_exciter=True,
    final_time=10.0,
)
ts_emt = read_timeseries_dpsim(f"logs/{log_emt}/{log_emt}.csv")

In [ ]:
def plot_three_domains(ts_sp, ts_dp, ts_emt, varname):
    """Overlay SP, DP, EMT on one plot; return (rmse_dp, rmse_emt) in % of nominal."""
    n = int((T_END - T_BEGIN) / T_STEP_COMMON)
    time = np.linspace(T_BEGIN, T_END, n)
    sv = ts_sp[varname].interpolate(T_STEP_COMMON).values[:n]
    dv = ts_dp[varname].interpolate(T_STEP_COMMON).values[:n]
    ev = ts_emt[varname].interpolate(T_STEP_COMMON).values[:n]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(time, sv, label="SP", color="C0")
    ax.plot(time, dv, "--", label="DP", color="C1")
    ax.plot(time, ev, ":", label="EMT", color="C2", linewidth=1.8)
    ax.set_xlabel("time (s)")
    ax.set_ylabel(ylabels.get(varname, varname))
    ax.set_title(titles.get(varname, varname))
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()
    nom = nominal_values.get(varname, 1.0)
    rmse_dp = np.sqrt(np.mean((sv - dv) ** 2)) / nom * 100
    rmse_emt = np.sqrt(np.mean((sv - ev) ** 2)) / nom * 100
    print(f"{varname}: RMSE(DP vs SP)={rmse_dp:.4f}%   RMSE(EMT vs SP)={rmse_emt:.4f}%")
    return rmse_dp, rmse_emt


rmse_dp_wr, rmse_emt_wr = plot_three_domains(ts_sp_exc, ts_dp, ts_emt, "w_r")
rmse_dp_Ef, rmse_emt_Ef = plot_three_domains(ts_sp_exc, ts_dp, ts_emt, "Ef")
rmse_dp_del, rmse_emt_del = plot_three_domains(ts_sp_exc, ts_dp, ts_emt, "delta")

In [ ]:
tol_cross = 0.1  # % for w_r
tol_cross_Ef = 1.0  # % for Ef  (exciter dynamics differ slightly between SP/DP/EMT)
tol_cross_del = 1.5  # % for delta

assert rmse_dp_wr < tol_cross, f"DP vs SP w_r RMSE {rmse_dp_wr:.4f}% > {tol_cross}%"
assert (
    rmse_dp_Ef < tol_cross_Ef
), f"DP vs SP Ef RMSE {rmse_dp_Ef:.4f}% > {tol_cross_Ef}%"
assert (
    rmse_dp_del < tol_cross_del
), f"DP vs SP delta RMSE {rmse_dp_del:.4f}% > {tol_cross_del}%"
assert rmse_emt_wr < tol_cross, f"EMT vs SP w_r RMSE {rmse_emt_wr:.4f}% > {tol_cross}%"
assert (
    rmse_emt_Ef < tol_cross_Ef
), f"EMT vs SP Ef RMSE {rmse_emt_Ef:.4f}% > {tol_cross_Ef}%"
assert (
    rmse_emt_del < tol_cross_del
), f"EMT vs SP delta RMSE {rmse_emt_del:.4f}% > {tol_cross_del}%"
print("Section 6 cross-domain assertions passed.")

## Section 7 — All Controllers Together (SP, DP, EMT)

Three controller combinations are tested:
- **Combo A**: Exciter + PSS + TurbineGovernorType1
- **Combo B**: Exciter + PSS + SteamTurbineGovernor
- **Combo C**: Exciter + PSS + HydroTurbineGovernor

Each combination is run in **SP, DP, and EMT** (9 simulations total).
Passing in SP does not guarantee correctness in DP/EMT — all three must be verified.

Smoke-test asserts (no PSAT reference for combined controllers):
- `Ef` finite and > 0 at t=10 s (exciter active)
- `Tm` finite and > 0 at t=10 s (governor active)
- `w_r` within 0.5% of 1.0 pu at t=10 s (stable frequency recovery)

In [ ]:
# All-controllers-together: 3 combos × 3 domains = 9 simulations
combos = {
    "ExcPSS_TGov1": dict(with_exciter=True, with_pss=True, with_governor_type1=True),
    "ExcPSS_Steam": dict(with_exciter=True, with_pss=True, with_steam_governor=True),
    "ExcPSS_Hydro": dict(with_exciter=True, with_pss=True, with_hydro_governor=True),
}

# results_sec7[combo_name][domain] = timeseries dict
results_sec7 = {}
for combo_name, kwargs in combos.items():
    results_sec7[combo_name] = {}
    for domain in ("SP", "DP", "EMT"):
        suffix = f"_Sec7_{combo_name}"
        log = smib_fault(
            sim_pf, gen_pf, domain=domain, name_suffix=suffix, final_time=10.0, **kwargs
        )
        results_sec7[combo_name][domain] = read_timeseries_dpsim(
            f"logs/{log}/{log}.csv"
        )
        print(f"  {combo_name}/{domain} done")

In [ ]:
# Visual overview: SP/DP/EMT overlay for each combo (Te, w_r, Ef, Tm)
for combo_name in combos:
    print(f"\n{'='*60}")
    print(f"Combo: {combo_name}")
    print("=" * 60)
    ts_s = results_sec7[combo_name]["SP"]
    ts_d = results_sec7[combo_name]["DP"]
    ts_e = results_sec7[combo_name]["EMT"]
    for var in ("Te", "w_r", "Ef", "Tm"):
        plot_three_domains(ts_s, ts_d, ts_e, var)

In [ ]:
# Smoke-test assertions: all 9 combinations must be numerically stable
wr_tol = 0.005  # 0.5% around 1.0 pu

for combo_name in combos:
    for domain in ("SP", "DP", "EMT"):
        ts = results_sec7[combo_name][domain]
        label = f"{combo_name}/{domain}"

        Ef_end = ts["Ef"].values[-1]
        assert (
            math.isfinite(Ef_end) and Ef_end > 0
        ), f"{label}: Ef not positive finite at t=10s: {Ef_end}"

        Tm_end = ts["Tm"].values[-1]
        assert (
            math.isfinite(Tm_end) and Tm_end > 0
        ), f"{label}: Tm not positive finite at t=10s: {Tm_end}"

        wr_end = ts["w_r"].values[-1]
        assert (
            abs(wr_end - 1.0) < wr_tol
        ), f"{label}: w_r {wr_end:.4f} pu not within 0.5% of 1.0 pu at t=10s"

        print(f"  {label:30s}  Ef={Ef_end:.4f}  Tm={Tm_end:.4f}  w_r={wr_end:.6f}")

print("\nSection 7 all-controllers-together assertions passed for SP, DP, EMT.")